# Face Denoising Autoencoder — from scratch 
### No frameworks. No shortcuts. Pure NumPy and Math.

Most people learn deep learning by calling `model.fit()` and watching a progress bar.
This project takes the opposite approach , every weight, every gradient, every update is computed by hand. The goal is not just to build something that works, but to deeply understand why it works.

-The task: given a corrupted image of a human face, reconstruct a cleaner version.
-The tool: a fully connected autoencoder trained with backpropagation, implemented entirely in NumPy.

### -Network architecture:

The network follows a symmetric hourglass shape an encoder that compresses and a decoder that reconstructs. 
The input image is a 128×128 grayscale face, flattened into a vector of 16,384 numbers. The encoder progressively reduce this through two hidden layers (2,048 then 512 neurons) down to a bottleneck of just 128 neurons. 

At this point the entire face is represented by 128 numbers with a compression ratio of 128:1.
The decoder then mirrors this process in reverse,expanding back through 512 and 2,048 neurons until it reaches 16,384 outputs which is 
the reconstructed face.


- **the ReLU activation function:**    $Relu(z) = max(0, z)$   is used in all hidden layers.

- **the Sigmoid function:**  $ \sigma(z) = \frac{1} {1 + e^{-z}} $ is used at the final output layer to keep pixel values between 0 and 1.

### neural network diagram:

<center>
<img src="neural_network.png" >
</center>

### now lets move  the real work :

In [ ]:
import numpy as np

In [ ]:
sigma  = lambda z : 1 / (1 + np.exp(-z))
d_sigma = lambda z : np.cosh(z/2)**(-2) / 4
relu = lambda z : np.maximum(0, z)
d_relu = lambda z : np.where(z > 0, 1, 0)

# This function initialises the network with it's structure, it also resets any training already done.
def reset_network (n1 = 2048, n2 = 512,n3=128,n4=512,n5=2048,n6=16384 ,random=np.random) :
    global W1, W2, W3, W4, W5, W6 ,b1, b2, b3, b4 , b5 , b6 
    W1 = random.randn(n1, n6) / 2
    W2 = random.randn(n2, n1) / 2
    W3 = random.randn(n3,n2) /2
    W4 =random.randn(n4,n3) /2
    W5 =random.randn(n5,n4) /2
    W6 =random.randn(n6,n5) /2
    b1 = random.randn(n1, 1) / 2
    b2 = random.randn(n2, 1) / 2
    b3 = random.randn(n3, 1) / 2
    b4 = random.randn(n4, 1) / 2
    b5 = random.randn(n5, 1) / 2
    b6 = random.randn(n6, 1) / 2

# This function feeds forward each activation to the next layer. It returns all weighted sums and activations.
def network_function(a0) :
    z1 = W1 @ a0 + b1
    a1 = relu(z1)
    z2 = W2 @ a1 + b2
    a2 = relu(z2)
    z3 = W3 @ a2 + b3
    a3 = relu(z3)
    z4 = W4 @ a3+b4
    a4 = relu(z4)
    z5 = W5 @ a4 + b5
    a5 = relu(z5)
    z6 = W6 @ a5 + b6
    a6 = sigma(z6)
    
    return a0, z1, a1, z2, a2, z3, a3,z4, a4, z5, a5, z6, a6


# This is the cost function of a neural network with respect to a training set.
def cost(x, y):
    return np.linalg.norm(network_function(x)[-1] - y)**2 / x.shape[1]





     

In [ ]:
#now we have to the jacobiam matrix for each  weight and bias , and do the backpropagation algorithm 

### Layer 6 — output layer (sigmoid)

$$\frac{\partial C}{\partial \mathbf{W}^{(6)}} =
\frac{\partial C}{\partial \mathbf{a}^{(6)}}
\frac{\partial \mathbf{a}^{(6)}}{\partial \mathbf{z}^{(6)}}
\frac{\partial \mathbf{z}^{(6)}}{\partial \mathbf{W}^{(6)}}$$

$$\frac{\partial C}{\partial \mathbf{b}^{(6)}} =
\frac{\partial C}{\partial \mathbf{a}^{(6)}}
\frac{\partial \mathbf{a}^{(6)}}{\partial \mathbf{z}^{(6)}}
\frac{\partial \mathbf{z}^{(6)}}{\partial \mathbf{b}^{(6)}}$$

Where:

$$\frac{\partial C}{\partial \mathbf{a}^{(6)}} = 2(\mathbf{a}^{(6)} - \mathbf{y})$$

$$\frac{\partial \mathbf{a}^{(6)}}{\partial \mathbf{z}^{(6)}} = \sigma'(\mathbf{z}^{(6)})$$

$$\frac{\partial \mathbf{z}^{(6)}}{\partial \mathbf{W}^{(6)}} = \mathbf{a}^{(5)}, \qquad \frac{\partial \mathbf{z}^{(6)}}{\partial \mathbf{b}^{(6)}} = 1$$


In [ ]:
# Jacobian for the sixth layer weights .
def J_W6 (x, y) :
    # First get all the activations and weighted sums at each layer of the network.
    a0, z1, a1, z2, a2, z3, a3,z4, a4, z5, a5, z6, a6= network_function(x)
    # We'll use the variable J to store parts of our result as we go along, updating it in each line.
    # Firstly, we calculate dC/da6, using the expressions above.
    J = 2 * (a6 - y)
    # Next we multiply the result we've calculated by the derivative of sigma, evaluated at z6.
    J = J * d_sigma(z6)
    # Then we take the dot product (along the axis that holds the training examples) with the final partial derivative,
    # i.e. dz6/dW6 = a5
    # and divide by the number of training examples, for the average over all training examples.
    J = J @ a5.T / x.shape[1]
    # Finally we get  the result out of the function.
    return J

def J_b6 (x, y) :
    # As last time, we'll first set up the activations.
    a0, z1, a1, z2, a2, z3, a3,z4, a4, z5, a5, z6, a6= network_function(x) 
    # Next we should implement the first two partial derivatives of the Jacobian.
    J = 2 * (a6 - y)
    J = J * d_sigma(z6)
    # here  we don't need to multiply by dz6/db6, because that is multiplying by 1.
    # We still need to sum over all training examples however.
    J = np.sum(J, axis=1, keepdims=True) /  x.shape[1]
    return J

### Layer 5 — hidden layer (ReLU)

$$\frac{\partial C}{\partial \mathbf{W}^{(5)}} =
\frac{\partial C}{\partial \mathbf{a}^{(6)}}
\frac{\partial \mathbf{a}^{(6)}}{\partial \mathbf{a}^{(5)}}
\frac{\partial \mathbf{a}^{(5)}}{\partial \mathbf{z}^{(5)}}
\frac{\partial \mathbf{z}^{(5)}}{\partial \mathbf{W}^{(5)}}$$

$$\frac{\partial C}{\partial \mathbf{b}^{(5)}} =
\frac{\partial C}{\partial \mathbf{a}^{(6)}}
\frac{\partial \mathbf{a}^{(6)}}{\partial \mathbf{a}^{(5)}}
\frac{\partial \mathbf{a}^{(5)}}{\partial \mathbf{z}^{(5)}}
\frac{\partial \mathbf{z}^{(5)}}{\partial \mathbf{b}^{(5)}}$$

Where:

$$\frac{\partial \mathbf{a}^{(6)}}{\partial \mathbf{a}^{(5)}} =
\sigma'(\mathbf{z}^{(6)}) \cdot \mathbf{W}^{(6)}$$

$$\frac{\partial \mathbf{a}^{(5)}}{\partial \mathbf{z}^{(5)}} = \text{ReLU}'(\mathbf{z}^{(5)}), \qquad \text{where} \quad \text{ReLU}'(z) = \begin{cases} 1 & z > 0 \\ 0 & z \leq 0 \end{cases}$$

$$\frac{\partial \mathbf{z}^{(5)}}{\partial \mathbf{W}^{(5)}} = \mathbf{a}^{(4)}, \qquad \frac{\partial \mathbf{z}^{(5)}}{\partial \mathbf{b}^{(5)}} = 1$$


In [ ]:
def J_W5(x, y) :
    #The first two lines are identical to in J_W3.
    a0, z1, a1, z2, a2, z3, a3,z4, a4, z5, a5, z6, a6= network_function(x)     
    J = 2 * (a6 - y)
    # the next two lines implement da6/da5, first d_relu and then W6.
    J = J * d_sigma(z6)
    J = (J.T @ W6).T
    # then the final lines are the same as in J_W6 but with the layer number bumped down.
    J = J * d_relu(z5)
    J = J @ a4.T / x.shape[1]
    return J

def J_b5 (x, y) :
    a0, z1, a1, z2, a2, z3, a3,z4, a4, z5, a5, z6, a6= network_function(x)  
    J = 2 * (a6 - y)
    J = J * d_sigma(z6)
    J =(J.T @ W6).T
    J =J*d_relu(z5)
    #we dont have to multiply with dz5/db5 beause its equale to 1 
    J = np.sum(J, axis=1, keepdims=True) / x.shape[1]
    return J

### Layer 4 — hidden layer (ReLU)

$$\frac{\partial C}{\partial \mathbf{W}^{(4)}} =
\frac{\partial C}{\partial \mathbf{a}^{(6)}}
\frac{\partial \mathbf{a}^{(6)}}{\partial \mathbf{a}^{(5)}}
\frac{\partial \mathbf{a}^{(5)}}{\partial \mathbf{a}^{(4)}}
\frac{\partial \mathbf{a}^{(4)}}{\partial \mathbf{z}^{(4)}}
\frac{\partial \mathbf{z}^{(4)}}{\partial \mathbf{W}^{(4)}}$$

$$\frac{\partial \mathbf{a}^{(5)}}{\partial \mathbf{a}^{(4)}} =
\text{ReLU}'(\mathbf{z}^{(5)}) \cdot \mathbf{W}^{(5)}$$

$$\frac{\partial \mathbf{a}^{(4)}}{\partial \mathbf{z}^{(4)}} = \text{ReLU}'(\mathbf{z}^{(4)}), \qquad
\frac{\partial \mathbf{z}^{(4)}}{\partial \mathbf{W}^{(4)}} = \mathbf{a}^{(3)}, \qquad
\frac{\partial \mathbf{z}^{(4)}}{\partial \mathbf{b}^{(4)}} = 1$$


In [ ]:
#same logic 
def J_W4(x, y):
    a0, z1, a1, z2, a2, z3, a3, z4, a4, z5, a5, z6, a6 = network_function(x)
    J = 2 * (a6 - y)
    J = J * d_sigma(z6)
    J = (J.T @ W6).T
    J = J * d_relu(z5)
    J = (J.T @ W5).T
    J = J * d_relu(z4)
    J = J @ a3.T / x.shape[1]   
    return J

def J_b4(x, y):                  
    a0, z1, a1, z2, a2, z3, a3, z4, a4, z5, a5, z6, a6 = network_function(x)
    J = 2 * (a6 - y)
    J = J * d_sigma(z6)
    J = (J.T @ W6).T
    J = J * d_relu(z5)
    J = (J.T @ W5).T
    J = J * d_relu(z4)
    J = np.sum(J, axis=1, keepdims=True) / x.shape[1]
    return J

### Layer 3 — bottleneck (ReLU)

$$\frac{\partial C}{\partial \mathbf{W}^{(3)}} =
\frac{\partial C}{\partial \mathbf{a}^{(6)}}
\left(
\frac{\partial \mathbf{a}^{(6)}}{\partial \mathbf{a}^{(5)}}
\frac{\partial \mathbf{a}^{(5)}}{\partial \mathbf{a}^{(4)}}
\frac{\partial \mathbf{a}^{(4)}}{\partial \mathbf{a}^{(3)}}
\right)
\frac{\partial \mathbf{a}^{(3)}}{\partial \mathbf{z}^{(3)}}
\frac{\partial \mathbf{z}^{(3)}}{\partial \mathbf{W}^{(3)}}$$

$$\frac{\partial \mathbf{a}^{(4)}}{\partial \mathbf{a}^{(3)}} =
\text{ReLU}'(\mathbf{z}^{(4)}) \cdot \mathbf{W}^{(4)}$$

$$\frac{\partial \mathbf{a}^{(3)}}{\partial \mathbf{z}^{(3)}} = \text{ReLU}'(\mathbf{z}^{(3)}), \qquad
\frac{\partial \mathbf{z}^{(3)}}{\partial \mathbf{W}^{(3)}} = \mathbf{a}^{(2)}, \qquad
\frac{\partial \mathbf{z}^{(3)}}{\partial \mathbf{b}^{(3)}} = 1$$


In [ ]:
def J_W3(x, y):
    a0, z1, a1, z2, a2, z3, a3, z4, a4, z5, a5, z6, a6 = network_function(x)
    J=2*(a6-y)      #dc/da6
    J=J*(d_sigma(z6))
    J = (J.T @ W6).T
    J=J*(d_relu(z5))
    J = (J.T @ W5).T
    J=J*(d_relu(z4))
    J = (J.T @ W4).T
     #right now we finish the the term between parentesis
    J=J*d_relu(z3)
    J=J@a2.T / x.shape[1]
    return J
def J_b3(x, y):
    a0, z1, a1, z2, a2, z3, a3, z4, a4, z5, a5, z6, a6 = network_function(x)
    J=2*(a6-y) 
    J=J*(d_sigma(z6))
    J = (J.T @ W6).T
    J=J*(d_relu(z5))
    J = (J.T @ W5).T
    J=J*(d_relu(z4))
    J = (J.T @ W4).T
    J=J*d_relu(z3)
    J = np.sum(J, axis=1, keepdims=True) / x.shape[1]
    return J
    

    
    

### Layer 2 — hidden layer (ReLU)

$$\frac{\partial C}{\partial \mathbf{W}^{(2)}} =
\frac{\partial C}{\partial \mathbf{a}^{(6)}}
\left(
\frac{\partial \mathbf{a}^{(6)}}{\partial \mathbf{a}^{(5)}}
\cdots
\frac{\partial \mathbf{a}^{(3)}}{\partial \mathbf{a}^{(2)}}
\right)
\frac{\partial \mathbf{a}^{(2)}}{\partial \mathbf{z}^{(2)}}
\frac{\partial \mathbf{z}^{(2)}}{\partial \mathbf{W}^{(2)}}$$

$$\frac{\partial \mathbf{a}^{(3)}}{\partial \mathbf{a}^{(2)}} =
\text{ReLU}'(\mathbf{z}^{(3)}) \cdot \mathbf{W}^{(3)}$$

$$\frac{\partial \mathbf{a}^{(2)}}{\partial \mathbf{z}^{(2)}} = \text{ReLU}'(\mathbf{z}^{(2)}), \qquad
\frac{\partial \mathbf{z}^{(2)}}{\partial \mathbf{W}^{(2)}} = \mathbf{a}^{(1)}, \qquad
\frac{\partial \mathbf{z}^{(2)}}{\partial \mathbf{b}^{(2)}} = 1$$

In [ ]:
def J_W2(x, y):
    a0, z1, a1, z2, a2, z3, a3, z4, a4, z5, a5, z6, a6 = network_function(x)
    J = 2 * (a6 - y)
    J = J * d_sigma(z6)
    J = (J.T @ W6).T
    J = J * d_relu(z5)
    J = (J.T @ W5).T
    J = J * d_relu(z4)
    J = (J.T @ W4).T
    J = J * d_relu(z3)
    J = (J.T @ W3).T
    J = J * d_relu(z2)
    J = J @ a1.T / x.shape[1]
    return J

def J_b2(x, y):
    a0, z1, a1, z2, a2, z3, a3, z4, a4, z5, a5, z6, a6 = network_function(x)
    J = 2 * (a6 - y)
    J = J * d_sigma(z6)
    J = (J.T @ W6).T
    J = J * d_relu(z5)
    J = (J.T @ W5).T
    J = J * d_relu(z4)
    J = (J.T @ W4).T
    J = J * d_relu(z3)
    J = (J.T @ W3).T
    J = J * d_relu(z2)
    J = np.sum(J, axis=1, keepdims=True) / x.shape[1]
    return J

### Layer 1 — hidden layer (ReLU)

$$\frac{\partial C}{\partial \mathbf{W}^{(1)}} =
\frac{\partial C}{\partial \mathbf{a}^{(6)}}
\left(
\frac{\partial \mathbf{a}^{(6)}}{\partial \mathbf{a}^{(5)}}
\cdots
\frac{\partial \mathbf{a}^{(2)}}{\partial \mathbf{a}^{(1)}}
\right)
\frac{\partial \mathbf{a}^{(1)}}{\partial \mathbf{z}^{(1)}}
\frac{\partial \mathbf{z}^{(1)}}{\partial \mathbf{W}^{(1)}}$$

$$\frac{\partial \mathbf{a}^{(2)}}{\partial \mathbf{a}^{(1)}} =
\text{ReLU}'(\mathbf{z}^{(2)}) \cdot \mathbf{W}^{(2)}$$

$$\frac{\partial \mathbf{a}^{(1)}}{\partial \mathbf{z}^{(1)}} = \text{ReLU}'(\mathbf{z}^{(1)}), \qquad
\frac{\partial \mathbf{z}^{(1)}}{\partial \mathbf{W}^{(1)}} = \mathbf{a}^{(0)}, \qquad
\frac{\partial \mathbf{z}^{(1)}}{\partial \mathbf{b}^{(1)}} = 1$$


In [ ]:
def J_W1(x, y):
    a0, z1, a1, z2, a2, z3, a3, z4, a4, z5, a5, z6, a6 = network_function(x)
    J = 2 * (a6 - y)
    J = J * d_sigma(z6)
    J = (J.T @ W6).T
    J = J * d_relu(z5)
    J = (J.T @ W5).T
    J = J * d_relu(z4)
    J = (J.T @ W4).T
    J = J * d_relu(z3)
    J = (J.T @ W3).T
    J = J * d_relu(z2)
    J = (J.T @ W2).T
    J = J * d_relu(z1)
    J = J @ a0.T / x.shape[1]
    return J


def J_b1(x, y):
    a0, z1, a1, z2, a2, z3, a3, z4, a4, z5, a5, z6, a6 = network_function(x)
    J = 2 * (a6 - y)
    J = J * d_sigma(z6)
    J = (J.T @ W6).T
    J = J * d_relu(z5)
    J = (J.T @ W5).T
    J = J * d_relu(z4)
    J = (J.T @ W4).T
    J = J * d_relu(z3)
    J = (J.T @ W3).T
    J = J * d_relu(z2)
    J = (J.T @ W2).T
    J = J * d_relu(z1)
    J = np.sum(J, axis=1, keepdims=True) / x.shape[1]
    return J

### The pattern every layer follows

Each time we go one layer deeper we add one more term:
$$\frac{\partial \mathbf{a}^{(L)}}{\partial \mathbf{a}^{(L-1)}} = \text{ReLU}'(\mathbf{z}^{(L)}) \cdot \mathbf{W}^{(L)}$$

This just  the **chain rule** applied repeatedly 

#### the next  step to do is to Build your training pairs our training pairs


## Data Pipeline

We use the LFW dataset (~13,000 real clean face photos) .

Each image is converted to grayscale, resized to 128×128, normalized to [0,1], and flattened to a (16384, 1) vector. We then artificially corrupt each clean images by adding random noise ,giving us (noisy, clean) training pairs.


The network receives the noisy version and learns to reconstruct the clean one.

In [ ]:
import os
from PIL import Image

In [ ]:
#making clean images to make training pairs 

list_of_images=[]
for folder, _, files in os.walk('/home/aamr_/My_Autoencoder_project /lfw-deepfunneled'):
    for file in files:
        full_path = os.path.join(folder, file)
        with Image.open(full_path) as img:
            img=img.convert('L') 
            img=img.resize((128,128))
            array= np.array(img) / 255   
            list_of_images.append(array)
#now we have to flattened each of the arrays in the list to a (16384, 1) vector 
X_clean = np.array(list_of_images)
X_clean = X_clean.reshape(16384, 13233)  
    



In [ ]:
#making corupted images to make traning pairs 
noise=np.random.randn(16384, 13233)*0.1
X_corupted=noise+X_clean
X_corupted=np.clip(X_corupted , 0, 1)





In [ ]:


learning_rate = 0.01 
"""
each weight moves 1% of the gradient per update .
because we want to make sure we are not gonna over shoot when adjusting the the weight and biases to minimize the cost function
"""
epochs = 100         #we  loop through all images 100 times
batch_size = 32      #we  you process 32 images at a time  
"""
epoch 1:
    batch 1 → images 0 to 31      → update weights
    batch 2 → images 32 to 63     → update weights
    batch 3 → images 64 to 95     → update weights
    ... until we finish all the 13233 pic 
so each image is gonna be see 100 times because we have 100 epochs 
"""

In [ ]:

reset_network() #resetting nework 
losse_record= []

for i in range(epochs):
    for j in range(0, 13233, batch_size):
        x = X_corupted[:, j:j+batch_size]
        y = X_clean[:, j:j+batch_size]
        
        loss = cost(x, y)
        losse_record.append(loss)
        
        W1 = W1 - learning_rate * J_W1(x, y)
        b1 = b1 - learning_rate * J_b1(x, y)
        W2 = W2 - learning_rate * J_W2(x, y)
        b2 = b2 - learning_rate * J_b2(x, y)
        W3 = W3 - learning_rate * J_W3(x, y)
        b3 = b3 - learning_rate * J_b3(x, y)
        W4 = W4 - learning_rate * J_W4(x, y)
        b4 = b4 - learning_rate * J_b4(x, y)
        W5 = W5 - learning_rate * J_W5(x, y)
        b5 = b5 - learning_rate * J_b5(x, y)
        W6 = W6 - learning_rate * J_W6(x, y)
        b6 = b6 - learning_rate * J_b6(x, y)
        
        
        
    
        
        